#### Here we understand some of the tricks related to attention

This is a collection from
1. Karpathy's Makemore series -> 
   1. It helps us understand how the attention matrix mathematical torch operations are kept, some basic pytorch helping code. 
   2. It helps understand how matrix multiplication helps us get to the right attention matrices


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [3]:
# B, T, C  -> Batch , Time , Channel
# Batch -> THe different batches of text we are training on 
# Time -> How we processed the same text, as a special training sequence (8 different characters, each predicting the next)
# Channel -> Is the feature embedding for each token

In [4]:
### Creating random sample
B, T, C = 4, 8, 2 ## Random example for understanding attention
x= torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

<p>
## Right now these are random tokens, we want tokens to communicate properly <br>
# 1st token can't look at anything, 2nd, only sees first, 3rd, sees 1 and 2<br>
#  We only see past<br>
# Simple way is that we just do an average of the past tokens <br>
</p>

#### Version -1 -> Simplest average but most non-efficient way

In [5]:
xbow = torch.zeros((B,T, C)) ## X- Bag of words, a weighted average representation of tokens from past learning

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [6]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [7]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

#### Version -2 -> Matrix Multiplication

In [8]:
torch.manual_seed(1337)
torch.ones((3,3)) # This function creates a series of ones
a = torch.tril(torch.ones(3, 3)) ## Torch lower diagonal of matrix elements are kept
a = a / torch.sum(a, 1 ,keepdim=True) ## Average per row
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [9]:
b = torch.randint(0,10, (3,2)).float()
c = a @ b
c

tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333]])

In [10]:
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)
## C is no moving average of A rows

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[5., 7.],
        [2., 0.],
        [5., 3.]])
--
c=
tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333]])


##### Setting it up for B,T,C example

Using matrix multiplication, we did weighted average for batch multiplication

In [11]:
weights1 = torch.tril(torch.ones(T, T))
weights1 = weights1 /torch.sum(weights1, 1, keepdim=True)
print(weights1.shape, x.shape)
xbow2 = weights1 @ x  # (T, T) @ (B, T, C) -> This is a batch multiply
# In batch multiply identical operations are done to entire batch
print(xbow2.shape)
torch.allclose(xbow, xbow2)



torch.Size([8, 8]) torch.Size([4, 8, 2])
torch.Size([4, 8, 2])


True

#### Version -3 -> Most Efficient, with softmax

In [12]:
wei = torch.zeros((T,T))
tril = torch.tril(torch.ones((T, T)))
wei = wei.masked_fill(tril==0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [13]:
wei= F.softmax(wei, dim=1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [14]:
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

Right now these are the weighted average and each seems same
But here these numbers will change, and we will see the importance of the words and relation between words

### Inspecting attention 

In [15]:
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

In [16]:
from attention import attention_head
# xb, yb = get_batch('train')
B, T, C = 4, 8, 32
# print(xb.shape, yb.shape)
x = torch.randn(B, T, C).to(device)

In [17]:
head_size = 16

In [18]:
x.shape

torch.Size([4, 8, 32])

In [19]:
learning = attention_head(C, T, head_size, x, device)

torch.Size([4, 8, 8]) torch.Size([4, 8, 16])


In [20]:
learning.shape

torch.Size([4, 8, 16])